In [2]:
import pandas as pd

# Load datasets
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

# Check shapes
print('Customers:', customers.shape)
print('Orders:', orders.shape)
print('Order Items:', order_items.shape)
print('Payments:', payments.shape)
print('Products:', products.shape)
print('Reviews:', reviews.shape)

Customers: (99441, 5)
Orders: (99441, 8)
Order Items: (112650, 7)
Payments: (103886, 5)
Products: (32951, 9)
Reviews: (99224, 7)


In [4]:
# Revenue per order
revenue = order_items.groupby('order_id').agg(
    revenue=('price', 'sum'),
    freight=('freight_value', 'sum')
).reset_index()

revenue['total_revenue'] = revenue['revenue'] + revenue['freight']

revenue.head()

,order_id,revenue,freight,total_revenue
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04


In [5]:
# Merge orders with customers and revenue
business = orders.merge(customers, on='customer_id', how='left')
business = business.merge(revenue, on='order_id', how='left')

business.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,revenue,freight,total_revenue
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,29.99,8.72,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,118.70,22.76,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,159.90,19.22,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,45.00,27.20,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,19.90,8.72,28.62


In [8]:
total_revenue = business['total_revenue'].sum()
total_orders = business['order_id'].nunique()
total_customers = business['customer_unique_id'].nunique()
avg_order_value = total_revenue / total_orders

print(f'Total Revenue: {total_revenue:,.2f}')
print(f'Total Orders: {total_orders:,}')
print(f'Total Customers: {total_customers:,}')
print(f'Average Order Value: {avg_order_value:,.2f}')

Total Revenue: 15,843,553.24
Total Orders: 99,441
Total Customers: 96,096
Average Order Value: 159.33


In [6]:
business.to_csv('../data/business_master.csv', index=False)
print('Saved successfully!')

Saved successfully!


In [10]:
business.shape 
business['order_status'].value_counts().head() 
business.isnull().sum().sort_values(ascending=False).head()

order_delivered_customer_date    2965
order_delivered_carrier_date     1783
revenue                           775
freight                           775
total_revenue                     775
dtype: int64